# Final-round experiment: fine-tune IndoBERT (indolem/indobert-base-uncased) on the pseudo-labelled reviews

Adapted from a GEMASTIK 2022 preparation notebook by Alfan F. Wicaksono (Fasilkom UI). This run was stopped during epoch 2.

## Persiapan Gemastik 2022
Oleh : Alfan F. Wicaksono (Fasilkom UI)

Pelatihan ini ditujukan untuk mempersiapkan tim menghadapi Final Gemastik 2022 Data Mining. Data berformat gabungan tekstual dan tabular. Di tutorial kali ini, kita akan menggunakan data yang dikonstruksi oleh tim Eryawan, Belati Jagad, dan Ghana Ahmada.

## Persiapan Library

In [ ]:
!pip install nlp-id

In [ ]:
!pip install emoji

In [ ]:
!pip install gensim

In [ ]:
!pip install imblearn

In [ ]:
!pip install scattertext

# pelajari https://github.com/JasonKessler/Scattertext-PyData
# https://github.com/JasonKessler/scattertext

In [ ]:
pip install seaborn

In [ ]:
from collections import Counter

from nlp_id.stopword import StopWord
from nlp_id.postag import PosTag
from nlp_id.lemmatizer import Lemmatizer

from gensim.models import TfidfModel
from gensim.models import LsiModel
from gensim.corpora import Dictionary

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score

from imblearn.over_sampling import ADASYN


import emoji
import csv
import string
import re
import pandas as pd
import numpy as np
import tensorflow as tf
import keras.backend as K

import scattertext as st
from IPython.display import IFrame
from IPython.core.display import display, HTML
from scattertext import CorpusFromPandas, produce_scattertext_explorer
display(HTML("<style>.container { width:98% !important; }</style>"))

import seaborn as sns # untuk visualisasi
import matplotlib.pyplot as plt
plt.style.use('classic')
%matplotlib inline

# potongan kode di bawah adalah agar gambar graph yang
# dihasilkan mempunyai kualitas yang cukup baik
sns.set(rc={"figure.dpi":100, 'savefig.dpi':300})
sns.set_context('notebook')
sns.set_style("ticks")

from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

<ipython-input-57-f61ac27a6eb2>:45: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('retina')


In [ ]:
def f1_score(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    recall = true_positives / (possible_positives + K.epsilon())
    return 2 * (precision * recall) / (precision + recall + K.epsilon())

start = 0

def plot_loss(history_dict):
    key1 = list(history_dict.keys())[0]
    key2 = list(history_dict.keys())[2]
    loss_values = history_dict[key1][start:]
    val_loss_values = history_dict[key2][start:]
    plt.plot(loss_values, "b-", label=key1)
    plt.plot(val_loss_values, "r--", label=key2)
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.show()
    print(key1, ": ", history_dict[key1][-1], key2, ": ", history_dict[key2][-1])


def plot_metric(history_dict):
    key1 = list(history_dict.keys())[1]
    key2 = list(history_dict.keys())[3]
    metric_values = history_dict[key1][start:]
    val_metric_values = history_dict[key2][start:]
    plt.plot(metric_values, "b-", label=key1)
    plt.plot(val_metric_values, "r--", label=key2)
    plt.title("Training vs Validation Metric")
    plt.legend()
    plt.show()
    print(key1, ": ", history_dict[key1][-1], key2, ": ", history_dict[key2][-1])

In [ ]:
!wget -c https://lumbung.cs.ui.ac.id/f/0a606ecd51d84a94b224/?dl=1 -O gemastik_22.csv

--2023-10-07 11:36:29--  https://lumbung.cs.ui.ac.id/f/0a606ecd51d84a94b224/?dl=1
Resolving lumbung.cs.ui.ac.id (lumbung.cs.ui.ac.id)... 152.118.148.86
Connecting to lumbung.cs.ui.ac.id (lumbung.cs.ui.ac.id)|152.118.148.86|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://lumbung.cs.ui.ac.id/seafhttp/files/b5d5b789-b41f-4be6-9435-1f00d8f10b81/subset.csv [following]
--2023-10-07 11:36:30--  https://lumbung.cs.ui.ac.id/seafhttp/files/b5d5b789-b41f-4be6-9435-1f00d8f10b81/subset.csv
Reusing existing connection to lumbung.cs.ui.ac.id:443.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [ ]:
data = pd.read_csv("train_iter4.csv")
data.head()

,source,review_id,user_name,review_title,review_description,rating,thumbs_up,review_date,developer_response,developer_response_date,appVersion,laguage_code,country_code,expedition,sentimen
0,Google Play,2c6973de-c75a-405d-9b9e-e5bf233f200a,Bayu Winalas,NaN,Hingga ini masih baik-baik aja 🥰🥰🥰,5,0,2021-09-02 03:36:26,NaN,NaN,3.13.1,id,id,J&T,1.0
1,Google Play,94fde3c7-4a37-4c92-875c-2f33d585f490,Wahyono 345,NaN,"Terima kasih JNE,selama ini selalu pakai jasa ...",5,1,2020-12-14 14:38:40,NaN,NaN,1.1.14,id,id,JNE,1.0
2,Google Play,31927262-6bd2-4ec5-abe5-757dc7df9a7e,Pengguna Google,NaN,ok lah bisa ngebantu,5,0,2019-03-12 15:29:01,NaN,NaN,3.1.1,id,id,J&T,1.0
3,Google Play,9f71ba44-d15c-438b-b3d0-22f861137f03,Pengguna Google,NaN,langganan,5,0,2019-02-28 00:50:52,NaN,NaN,3.1.1,id,id,J&T,1.0
4,Google Play,b409cdd4-1998-4d48-a486-8a5846e446f4,Pengguna Google,NaN,T O P BGT . . Pengiriman super cepat bgt. .suk...,5,0,2016-05-04 12:17:49,NaN,NaN,NaN,id,id,J&T,1.0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109395 entries, 0 to 109394
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   source                   109395 non-null  object 
 1   review_id                109395 non-null  object 
 2   user_name                109395 non-null  object 
 3   review_title             0 non-null       float64
 4   review_description       109395 non-null  object 
 5   rating                   109395 non-null  int64  
 6   thumbs_up                109395 non-null  int64  
 7   review_date              109395 non-null  object 
 8   developer_response       1890 non-null    object 
 9   developer_response_date  1890 non-null    object 
 10  appVersion               78291 non-null   object 
 11  laguage_code             109395 non-null  object 
 12  country_code             109395 non-null  object 
 13  expedition               109395 non-null  object 
 14  sent

### BERT-Based Models

#### Install Transformers

In [ ]:
!pip install transformers

#### Loading BERT Model

In [ ]:
pip install torch

In [ ]:
if tf.config.experimental.list_physical_devices('GPU'):
    print("GPU is available")
else:
    print("GPU is not available")

GPU is available


In [ ]:
X = data['review_description'].values
y = np.array(data["sentimen"])
print(X.shape, y.shape)
y[:10]

(109395,) (109395,)


array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((87516,), (21879,), (87516,), (21879,))

In [ ]:
from transformers import AutoTokenizer, TFBertModel

# MODEL = 'ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa'
MODEL = "indolem/indobert-base-uncased"
# MODEL = "w11wo/indonesian-roberta-base-sentiment-classifier"

tokenizer = AutoTokenizer.from_pretrained(MODEL)


bert_model = TFBertModel.from_pretrained(MODEL, from_pt=True)


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predict

In [ ]:
X = data['review_description'].values
y = np.array(data["sentimen"])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

MAX_LENGTH = 128

train_encodings = tokenizer(
    text=X_train.tolist(),
    add_special_tokens=True,
    max_length=MAX_LENGTH,
    truncation=True,
    padding=True,
    return_tensors='tf',
    # return_tensors='pt',
    return_token_type_ids=False,
    return_attention_mask=True,
    verbose=True
)

val_encodings = tokenizer(
    text=X_val.tolist(),
    add_special_tokens=True,
    max_length=MAX_LENGTH,
    truncation=True,
    padding=True,
    return_tensors='tf',
    # return_tensors='pt',
    return_token_type_ids=False,
    return_attention_mask=True,
    verbose=True
)

token_ids_train = np.array(train_encodings["input_ids"])
token_masks_train = np.array(train_encodings["attention_mask"])
token_ids_val = np.array(val_encodings["input_ids"])
token_masks_val = np.array(val_encodings["attention_mask"])

In [ ]:
pip install evaluate

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification, TFBertModel, TrainingArguments, Trainer, \
    AutoModelForSequenceClassification, EarlyStoppingCallback
import evaluate
import torch

import tensorflow as tf
import keras.backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense

print(torch.cuda.is_available())
print(tf.config.list_physical_devices('GPU'))

True
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


#### BERT's Tokenizer

In [ ]:
weight_initializer = tf.keras.initializers.GlorotNormal(seed=42)

input_ids = Input(shape=(MAX_LENGTH,), dtype=tf.int32, name="input_ids")
input_mask = Input(shape=(MAX_LENGTH,), dtype=tf.int32, name="attention_mask")

embeddings = bert_model(input_ids, attention_mask=input_mask)[0]  # 0 = last hidden state, 1 = poller_output
out = tf.keras.layers.GlobalMaxPool1D()(embeddings)
out = Dense(128, activation='relu', kernel_initializer = weight_initializer)(out)
out = tf.keras.layers.Dropout(0.1)(out)
out = Dense(32, activation='relu', kernel_initializer = weight_initializer)(out)

y = Dense(2, activation='softmax', kernel_initializer=weight_initializer)(out)

model = tf.keras.Model(inputs=[input_ids, input_mask], outputs=y)
model.layers[2].trainable = True

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=5e-05,
    decay_steps=10000,
    decay_rate=0.01)

optimizer = Adam(
    learning_rate=5e-05,  # HF recommendation
    epsilon=1e-08,
    clipnorm=1.0
)
loss = CategoricalCrossentropy()
metric = f1_score
# metric = CategoricalAccuracy('balanced_accuracy')

model.compile(
    optimizer=optimizer,
    loss=loss,
    # loss="categorical_crossentropy",
    metrics=metric
)

with tf.device('/device:GPU:0'):
    history = model.fit(
        x={'input_ids': token_ids_train, 'attention_mask': token_masks_train},
        y=to_categorical(y_train),
        validation_data=({'input_ids': token_ids_val, 'attention_mask': token_masks_val  },
                         to_categorical(y_val)),
        epochs=5,
        batch_size=8,
    )

Epoch 1/5


9572/9572 [==============================] - 2792s 282ms/step - loss: 0.4706 - f1_score: 0.7930 - val_loss: 1.0873 - val_f1_score: 0.6384
Epoch 2/5
4187/9572 [============>.................] - ETA: 21:19 - loss: 0.4752 - f1_score: 0.7860

KeyboardInterrupt: ignored